# Daily MinV panel

UTC-day MinV vs EGARCH 5%/1% bands on HL + Deribit + Kraken. Paper Table-2 style post-trough returns + \(h_n\) fragility + time-split falsifier.


In [ ]:
from pathlib import Path
import json
import pandas as pd
import numpy as np
OUT = Path('../../out/daily_minv').resolve()
daily = json.loads((OUT / 'daily_minv_eth.json').read_text())
df = pd.DataFrame([r for r in daily['rows'] if r.get('ok')])
print('symbol', daily.get('symbol'), 'days', daily.get('days'))
print('n_ok', len(df), 'n_complete', int(df['complete'].sum()) if len(df) else 0)
print('sig5', int(df['sig_5'].sum()) if len(df) else 0, 'sig5&complete', int((df['sig_5'] & df['complete']).sum()) if len(df) else 0)
days = sorted(df['day'].unique())
mid = len(days)//2
early, late = set(days[:mid]), set(days[mid:])
print('time-split early', sorted(early), 'sig', int(df[df.day.isin(early) & df.sig_5].shape[0]))
print('time-split late ', sorted(late), 'sig', int(df[df.day.isin(late) & df.sig_5].shape[0]))
print(df.pivot_table(index='venue', columns='hn_min', values='min_v', aggfunc='mean').round(2))


### MinV panel

`fig_minv_panel.png`


In [ ]:
from IPython.display import Image, display
from pathlib import Path
display(Image(filename=str(Path('../../out/daily_minv/fig_minv_panel.png'))))


### \(h_n\) fragility

`fig_hn_fragility.png`


In [ ]:
from IPython.display import Image, display
from pathlib import Path
p = Path('../../out/daily_minv/fig_hn_fragility.png')
if p.exists():
    display(Image(filename=str(p)))


### Post-trough returns

`fig_post_trough.png`


In [ ]:
from IPython.display import Image, display
from pathlib import Path
p = Path('../../out/daily_minv/fig_post_trough.png')
if p.exists():
    display(Image(filename=str(p)))


## Gate

- `risk.daily_minv_panel` → **Promote** if ≥1 complete-day sig@5% appears in **both** early and late halves.
- `info.post_trough_ret` → **Promote** only if mean post-ret CI excludes 0 with time-split; else Hold.
